In [ ]:
import pandas as pd

df = pd.read_csv("../data/sample.csv")
df["cats"] = df["categories"].str.split()
df["primary"] = df["cats"].str[0]
df["field"] = df["primary"].str.split(".").str[0]
df["n_cats"] = df["cats"].str.len()
df["abs_len"] = df["abstract"].str.split().str.len()

print(df.shape)
print(df["field"].value_counts().head(25))
print(df["n_cats"].value_counts().sort_index())
print(df["abs_len"].describe())

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

train = pd.read_csv("../data/train.csv")
val = pd.read_csv("../data/val.csv")

vec = TfidfVectorizer(max_features=50000, sublinear_tf=True, stop_words="english")
X_train = vec.fit_transform(train["text"])
X_val = vec.transform(val["text"])
clf = LogisticRegression(max_iter =1000).fit(X_train, train["label"])
pred = clf.predict(X_val)
cm = confusion_matrix(val["label"],pred)
df_cm  = pd.DataFrame(cm, index = clf.classes_, columns = clf.classes_)
val["pred"] = pred
#print(accuracy_score(val["label"],pred))
#print(cm)
wrong_eess = val[(val["label"]=="eess")&(val["label"]!= val["pred"])]
clf_bal = LogisticRegression(class_weight = "balanced",max_iter = 1000).fit(X_train, train["label"])
pred_bal = clf_bal.predict(X_val)
#print(accuracy_score(val["label"],pred_bal))
#print(f1_score(val["label"],pred,average = "macro"))
#print(f1_score(val["label"],pred_bal,average = "macro"))
#print(classification_report(val["label"], pred_bal, digits=3))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(val["label"],pred_bal,xticks_rotation = "vertical",normalize="true",values_format=".2f",ax=ax)
plt.savefig("../results/confusion_matrix_balanced.png", bbox_inches="tight", dpi=150)